## EDA

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Посмотрим на пропуски в датасетах 


In [3]:
# Используем sep=None для автоопределения разделителя и чистим шапку
base_df = pd.read_csv('data/raw/liquidity_base.csv', sep=None, engine='python')
base_df.columns = base_df.columns.str.strip()

cnt_df = pd.read_csv('data/raw/liquidity_cnt.csv', sep=None, engine='python')
cnt_df.columns = cnt_df.columns.str.strip()

print("=== АНАЛИЗ ПРОПУСКОВ (NaN) В БАЗЕ ===")
missing_stats = base_df.isna().sum()
missing_percent = (missing_stats / len(base_df)) * 100
missing_df = pd.DataFrame({'Пропуски': missing_stats, 'Доля (%)': missing_percent})
print(missing_df[missing_df['Пропуски'] > 0].sort_values(by='Доля (%)', ascending=False).round(2))
print("-" * 50)


print("\n=== ДЕТЕКТОР АНОМАЛИЙ В BASE_DF ===")
# 1. Аномалии цены
mask_price_low = base_df['price'] < 30000
mask_price_high = base_df['price'] > 20000000 # Порог 20 млн 
print(f"Авто дешевле 30к руб: {mask_price_low.sum()} шт.")
print(f"Авто дороже 20 млн руб: {mask_price_high.sum()} шт.")

# 2. Аномалии пробега
mask_mileage_high = base_df['mileage'] > 1000000
# Пробег < 100 км для авто старше 3 лет (считаем текущий год как максимальный в базе)
current_year = base_df['year'].max()
mask_mileage_fake = (base_df['mileage'] < 100) & (base_df['year'] < current_year - 2)
print(f"Пробег > 1 млн км: {mask_mileage_high.sum()} шт.")
print(f"Скрученный пробег (старые авто < 100 км): {mask_mileage_fake.sum()} шт.")

# 3. Аномалии оценки рынка (если imv не NaN)
# Избегаем деления на ноль
safe_imv = base_df['imv'].replace(0, np.nan)
delta_p = (base_df['price'] - safe_imv) / safe_imv
mask_overpriced = delta_p > 2.0 # Дороже рынка более чем на 200%
print(f"Цена завышена более чем на 200% от IMV: {mask_overpriced.sum()} шт.")
print("-" * 50)


print("\n=== ДЕТЕКТОР АНОМАЛИЙ В ЛОГАХ (CNT_DF) ===")
mask_bot_traffic = cnt_df['cnt_contacts'] > 500
print(f"Дней с аномальным трафиком (> 500 контактов): {mask_bot_traffic.sum()} шт.")

=== АНАЛИЗ ПРОПУСКОВ (NaN) В БАЗЕ ===
                Пропуски  Доля (%)
rating            105405     25.53
reviews           105405     25.53
equipment          95499     23.13
imv                69118     16.74
removal_reason     61625     14.93
modification         714      0.17
generation           161      0.04
--------------------------------------------------

=== ДЕТЕКТОР АНОМАЛИЙ В BASE_DF ===
Авто дешевле 30к руб: 439 шт.
Авто дороже 20 млн руб: 848 шт.
Пробег > 1 млн км: 0 шт.
Скрученный пробег (старые авто < 100 км): 1904 шт.
Цена завышена более чем на 200% от IMV: 748 шт.
--------------------------------------------------

=== ДЕТЕКТОР АНОМАЛИЙ В ЛОГАХ (CNT_DF) ===
Дней с аномальным трафиком (> 500 контактов): 30 шт.


### 1. Что делаем с аномалиями (Выбросы)

* Цена < 30к (439 шт): Сносим к чертовой матери. Это металлолом, продажа ПТС или мошенники. 
* Цена > 20 млн (848 шт): Оставляем, раз ты настаиваешь, но перед кластеризацией будем использовать логарифм цены (np.log1p(price)), иначе они утащат на себя центроиды.
* Скрученный пробег (1904 шт): Сносим. Это 0.4% от всей базы (у вас там около 412к строк судя по процентам). Нет смысла тратить время на предсказание их реального пробега, это просто грязный шум от перекупов.
* Неадекватная цена $\Delta P > 2$ (748 шт): Сносим. Чуваки, продающие тачки в 3 раза дороже рынка, не получат контактов просто из-за своей жадности. Это не системная проблема, это клиника.
* Боты в логах (> 500 контактов за день): Мы не удаляем эти дни, иначе порвем временные ряды (ту самую плотную сетку). Мы их «клипаем» (capping) — приравниваем все аномальные значения к 99-му перцентилю (например, к 50 контактам).

### 2. Что делаем с пропусками (NaN)

Здесь нужна жесткая продуктовая логика. Пропуск — это часто не ошибка, а сигнал.

* `rating` и `reviews` (25.53%): Это новые продавцы. У них нет рейтинга. Заполняем reviews нулем, а rating — маркером -1 (или 0). Алгоритм сам поймет, что "минус первый" рейтинг — это каста новичков, и у них ликвидность обычно ниже из-за недоверия.
* `equipment` (23.13%): Огромный кусок. Лень продавца расписывать комплектацию — это шикарная фича. Заменяем NaN на строковое значение "Не указано". Вы потом на защите покажете, что объявления без описания комплектации теряют 30% звонков.
* `imv` (16.74%): Это рыночная оценка, критически важная для расчета выгоды. Выкидывать 16% базы — непозволительная роскошь. Мы заполним пустые imv медианной ценой машин той же марки, модели и года. 
* `removal_reason` (14.93%): Причина снятия. Она вообще не должна участвовать в предсказании ликвидности, потому что на момент публикации объявления мы её не знаем (это заглядывание в будущее, data leak). Заполняем "Неизвестно".
* `modification` и `generation` (<0.2%): Удаляем эти строки. Их слишком мало, чтобы возиться.

In [4]:
base_df = pd.read_csv('data/raw/liquidity_base.csv', sep=None, engine='python')
base_df.columns = base_df.columns.str.strip()

cnt_df = pd.read_csv('data/raw/liquidity_cnt.csv', sep=None, engine='python')
cnt_df.columns = cnt_df.columns.str.strip()

print(f"Исходный размер базы: {base_df.shape[0]}")

# ==========================================
# ШАГ 1: УДАЛЕНИЕ МУСОРА И АНОМАЛИЙ
# ==========================================
# 1. Сносим дешевый мусор (< 30к)
base_df = base_df[base_df['price'] >= 30000]

# 2. Сносим скрученные пробеги (< 100 км для авто старше 2 лет)
current_year = base_df['year'].max()
fake_mileage_mask = (base_df['mileage'] < 100) & (base_df['year'] < current_year - 2)
base_df = base_df[~fake_mileage_mask]

# 3. Сносим строки с микро-пропусками
base_df = base_df.dropna(subset=['modification', 'generation'])

# ==========================================
# ШАГ 2: УМНОЕ ЗАПОЛНЕНИЕ ПРОПУСКОВ (IMPUTATION)
# ==========================================
# 1. Рейтинг и отзывы (Новички)
base_df['rating'] = base_df['rating'].fillna(-1.0)
base_df['reviews'] = base_df['reviews'].fillna(0)

# 2. Комплектация (Лень продавца)
base_df['equipment'] = base_df['equipment'].fillna('Не указано')

# 3. Причина снятия
base_df['removal_reason'] = base_df['removal_reason'].fillna('Неизвестно')

# 4. Восстановление IMV (Рыночная оценка)
# Вычисляем медианную цену для каждой связки "Марка-Модель-Год"
imv_fill = base_df.groupby(['brand', 'model', 'year'])['price'].transform('median')
# Если IMV пустой, берем медиану. Если медианы нет (редкая машина), берем саму цену (delta будет 0)
base_df['imv'] = base_df['imv'].fillna(imv_fill).fillna(base_df['price'])

# ==========================================
# ШАГ 3: ФИЛЬТРАЦИЯ ЖАДНЫХ ПРОДАВЦОВ ПОСЛЕ ВОССТАНОВЛЕНИЯ IMV
# ==========================================
# Теперь, когда IMV есть у всех, сносим тех, кто хочет х3 от рынка
base_df['delta_p'] = (base_df['price'] - base_df['imv']) / base_df['imv']
base_df = base_df[base_df['delta_p'] <= 2.0]

print(f"Размер базы после жесткой чистки: {base_df.shape[0]}")

# ==========================================
# ШАГ 4: ЧИСТКА ЛОГОВ (CAPPING)
# ==========================================
# Клипаем ботов на 99-м перцентиле, чтобы не рвать дни тишины
p99 = cnt_df['cnt_contacts'].quantile(0.99)
cnt_df['cnt_contacts'] = np.where(cnt_df['cnt_contacts'] > p99, p99, cnt_df['cnt_contacts'])

# Синхронизируем логи с очищенной базой (выкидываем из логов удаленные машины)
valid_ids = base_df['id'].unique()
cnt_df = cnt_df[cnt_df['id'].isin(valid_ids)]

# Сохраняем очищенные "сырые" данные, чтобы потом собирать из них витрины
base_df.to_csv('data/processed/clean_base.csv', index=False)
cnt_df.to_csv('data/processed/clean_cnt.csv', index=False)


Исходный размер базы: 412801
Размер базы после жесткой чистки: 408734


### Методология формирования аналитических витрин

Для решения задач хакатона была разработана архитектура данных, разделенная на два уровня агрегации. Это позволяет корректно учитывать как статические характеристики лотов, так и динамику пользовательского интереса.

#### 1. Статическая витрина (Факторы ликвидности)

Целевая переменная для каждого объявления $i$ (суммарная ликвидность) определяется как интегральный показатель контактов за весь период экспозиции:

$$L_{i} = \sum_{t=1}^{T_{i}} C_{i, t}$$

Где:
- $L_{i}$ — совокупная ликвидность лота;
- $C_{i, t}$ — количество контактов по объявлению в день $t$;
- $T_{i}$ — общее время жизни объявления в системе.

В случае отсутствия лога активности для конкретного $ID$, значение $L_{i}$ принимается равным $0$. Данная витрина используется для выявления ключевых характеристик автомобиля (цена, пробег, регион), влияющих на итоговый спрос.

#### 2. Динамическая витрина (Анализ временных рядов и VAS)

Для поиска оптимального момента подключения платных услуг (VAS) необходимо анализировать форму кривой затухания спроса. Чтобы избежать смещения оценок из-за "пропусков" в данных (дней без контактов), произведено развертывание векторов времени в плотную сетку $\mathbf{T}_{i} = \{1, 2, \dots, T_{i}\}$. 

Функция ежедневной ликвидности $f(t)$ для объявления $i$ формализована следующим образом:

$$f(t) = 
\begin{cases} 
x_{t}, & \text{при наличии записи в логе в день } t \\
0, & \text{в противном случае}
\end{cases}$$

Такой подход позволяет корректно рассчитывать производную функции спроса и определять точку $t_{opt}$, в которой падение органического интереса требует применения инструментов продвижения.

### отсекаем 10 перцентиль, чтобы не оценивать редкие модели, берем фокус на самых популярных моделях

In [5]:
# 1. Загружаем уже первично очищенные данные
base_df = pd.read_csv('data/processed/clean_base.csv')
cnt_df = pd.read_csv('data/processed/clean_cnt.csv')

print(f"Размер базы ДО среза марок: {base_df.shape[0]} строк")
print(f"Потребление памяти ДО: {base_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ==========================================
# ШАГ 1: ФОКУС НА МАСС-МАРКЕТ (Scope Limitation)
# ==========================================
# Считаем частоту каждой марки
brand_counts = base_df['brand'].value_counts()

# Берем порог 90-го перцентиля. 
# Это значит, что мы отсекаем 90% марок, которые встречаются реже всего (длинный хвост),
# и оставляем топ-10% марок, которые генерируют основной объем рынка.
threshold = brand_counts.quantile(0.90)

# Список ходовых марок
top_brands = brand_counts[brand_counts >= threshold].index

# Фильтруем базу
base_df = base_df[base_df['brand'].isin(top_brands)]

# Обязательно чистим логи, чтобы там не остались контакты от удаленных редких тачек
valid_ids = base_df['id'].unique()
cnt_df = cnt_df[cnt_df['id'].isin(valid_ids)]

# ==========================================
# ШАГ 2: ОПТИМИЗАЦИЯ ПАМЯТИ (Тип Category)
# ==========================================
# Находим все колонки со строковым типом (object)
object_cols = base_df.select_dtypes(include=['object']).columns

# Конвертируем в category
for col in object_cols:
    base_df[col] = base_df[col].astype('category')

print("-" * 40)
print(f"Размер базы ПОСЛЕ среза марок: {base_df.shape[0]} строк")
print(f"Потребление памяти ПОСЛЕ: {base_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Перезаписываем файлы. Теперь они идеальны для сборки витрин.
base_df.to_csv('data/processed/clean_base.csv', index=False)
cnt_df.to_csv('data/processed/clean_cnt.csv', index=False)

Размер базы ДО среза марок: 408734 строк
Потребление памяти ДО: 503.00 MB


C:\Users\User\AppData\Local\Temp\ipykernel_1808\599634877.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = base_df.select_dtypes(include=['object']).columns


----------------------------------------
Размер базы ПОСЛЕ среза марок: 364861 строк
Потребление памяти ПОСЛЕ: 90.05 MB


In [8]:
def build_static_mart(base_df: pd.DataFrame, cnt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает статическую витрину: 1 объявление = 1 строка с итоговым количеством контактов.
    
    Args:
        base_df: Очищенный датафрейм с характеристиками авто.
        cnt_df: Очищенный датафрейм с логами контактов.
        
    Returns:
        pd.DataFrame: Готовая витрина для обучения моделей (LightGBM и др.).
    """
    total_contacts = cnt_df.groupby('id', as_index=False)['cnt_contacts'].sum()
    total_contacts.rename(columns={'cnt_contacts': 'total_contacts'}, inplace=True)

    df_task1 = base_df.merge(total_contacts, on='id', how='left')
    df_task1['total_contacts'] = df_task1['total_contacts'].fillna(0)
    
    return df_task1


def build_dynamic_mart(base_df: pd.DataFrame, cnt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает динамическую витрину с плотной временной сеткой (векторизованный метод).
    
    Args:
        base_df: Очищенный датафрейм с характеристиками авто.
        cnt_df: Очищенный датафрейм с логами контактов.
        
    Returns:
        pd.DataFrame: Временные ряды (Time-to-Sell) без пропусков по дням.
    """
    # 1. Расчет времени жизни с использованием векторизованного clip
    max_date = base_df['close_time'].max()
    close_filled = base_df['close_time'].fillna(max_date)
    
    lifetime_days = (close_filled - base_df['start_time']).dt.days
    base_df['lifetime_days'] = lifetime_days.fillna(1).clip(lower=1).astype(int)

    # 2. Векторизованная генерация плотной сетки (БЕЗ ЦИКЛОВ)
    # Размножаем индексы пропорционально lifetime_days
    grid = base_df[['id', 'lifetime_days']].loc[base_df.index.repeat(base_df['lifetime_days'])].reset_index(drop=True)
    
    # Восстанавливаем номер дня: кумулятивный счетчик внутри каждого id
    grid['day'] = grid.groupby('id').cumcount() + 1
    grid = grid.drop(columns=['lifetime_days'])

    # 3. Мерж с логами и заполнение дней без контактов нулями
    df_dynamics = grid.merge(cnt_df, on=['id', 'day'], how='left')
    df_dynamics['cnt_contacts'] = df_dynamics['cnt_contacts'].fillna(0)

    # 4. Финальное обогащение характеристиками авто
    df_task2 = df_dynamics.merge(
        base_df.drop(columns=['lifetime_days']), 
        on='id', 
        how='inner'
    )
    
    return df_task2


def main():
    """Основной пайплайн сборки витрин."""
    print("Загрузка очищенных данных")
    base_df = pd.read_csv('data/processed/clean_base.csv')
    base_df['start_time'] = pd.to_datetime(base_df['start_time'], errors='coerce')
    base_df['close_time'] = pd.to_datetime(base_df['close_time'], errors='coerce')
    
    cnt_df = pd.read_csv('data/processed/clean_cnt.csv')

    print("Сборка статической витрины (Task 1)")
    df_static = build_static_mart(base_df, cnt_df)
    df_static.to_csv('data/processed/processed_task1_total_liquidity.csv', index=False)
    print(f"Готово. Размер: {df_static.shape}")

    print("Сборка плотной динамической витрины (Task 2 & 3)")
    df_dynamic = build_dynamic_mart(base_df, cnt_df)
    df_dynamic.to_csv('data/processed/processed_task2_dense_dynamics.csv', index=False)
    print(f"Готово. Размер: {df_dynamic.shape}")

if __name__ == "__main__":
    main()

Загрузка очищенных данных
Сборка статической витрины (Task 1)
Готово. Размер: (364861, 25)
Сборка плотной динамической витрины (Task 2 & 3)
Готово. Размер: (7742928, 27)


In [ ]:
import pandas as pd
import numpy as np


# Настройка строгого стиля для презентации
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

# Загружаем собранные витрины
df_stat = pd.read_csv('data/processed/processed_task1_total_liquidity.csv')
df_dyn = pd.read_csv('data/processed/processed_task2_dense_dynamics.csv')

# ==========================================
# 1. КОРРЕЛЯЦИОННАЯ МАТРИЦА (Критерий жюри)
# ==========================================
plt.figure(figsize=(10, 8))
numeric_cols = ['price', 'imv', 'delta_p', 'mileage', 'year', 'total_contacts']
corr_matrix = df_stat[numeric_cols].corr(method='spearman')
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('График 1: Ранговая корреляция факторов ликвидности')
plt.tight_layout()
plt.savefig('data/processed/viz_1_correlation.png')
plt.close()

# ==========================================
# 2. ВЛИЯНИЕ ЖАДНОСТИ ПРОДАВЦА (Scatter plot)
# ==========================================
# Показываем, как переоценка (delta_p) убивает звонки
plt.figure(figsize=(10, 6))
# Берем случайные 10k точек, чтобы не перегружать график
sample_df = df_stat.sample(min(10000, len(df_stat)), random_state=42)
sns.scatterplot(x='delta_p', y='total_contacts', data=sample_df, alpha=0.3, edgecolor=None)
plt.axvline(0, color='red', linestyle='--', label='Рыночная цена (IMV)')
plt.title('График 2: Падение ликвидности при завышении цены (Overprice)')
plt.xlabel('Отклонение от рынка (delta_p)')
plt.ylabel('Суммарные контакты')
plt.legend()
plt.tight_layout()
plt.savefig('data/processed/viz_2_overprice.png')
plt.close()

# ==========================================
# 3. ПОИСК СЕГМЕНТОВ (Boxplot)
# ==========================================
# Для примера разобьем машины на 3 базовых сегмента по возрасту
df_stat['age_segment'] = pd.cut(
    df_stat['year'], 
    bins=[1900, 2010, 2019, 2025], 
    labels=['Старые (>15 лет)', 'Средние (5-15 лет)', 'Свежие (<5 лет)']
)

plt.figure(figsize=(10, 6))
sns.boxplot(x='age_segment', y='total_contacts', data=df_stat, showfliers=False)
plt.title('График 3: Сравнение ликвидности по возрастным сегментам')
plt.ylabel('Контакты (без выбросов)')
plt.tight_layout()
plt.savefig('data/processed/viz_3_segments_boxplot.png')
plt.close()

# ==========================================
# 4. ТИПЫ КРИВЫХ НАКОПЛЕНИЯ КОНТАКТОВ (Критерий жюри)
# ==========================================
# Мержим сегменты в динамическую витрину для графика
df_dyn = df_dyn.merge(df_stat[['id', 'age_segment']], on='id', how='left')

plt.figure(figsize=(12, 6))
# Считаем среднее количество контактов в каждый день для каждого сегмента
avg_curves = df_dyn.groupby(['age_segment', 'day'])['cnt_contacts'].mean().reset_index()
sns.lineplot(x='day', y='cnt_contacts', hue='age_segment', data=avg_curves[avg_curves['day'] <= 20])
plt.title('График 4: Профили затухания интереса (Time-to-Sell)')
plt.xlabel('День жизни объявления')
plt.ylabel('Среднее кол-во контактов')
plt.xlim(1, 20)
plt.tight_layout()
plt.savefig('data/processed/viz_4_time_curves.png')
plt.close()

# ==========================================
# 5. СРАВНЕНИЕ ТОП-МАРОК МАСС-МАРКЕТА
# ==========================================
plt.figure(figsize=(12, 6))
top_5_brands = df_stat['brand'].value_counts().nlargest(20).index
df_top_brands = df_stat[df_stat['brand'].isin(top_5_brands)]
sns.barplot(x='brand', y='total_contacts', data=df_top_brands, estimator=np.median, errorbar=None)
plt.title('График 5: Медианная ликвидность Топ-5 марок масс-маркета')
plt.ylabel('Медиана контактов')
plt.tight_layout()
plt.savefig('data/processed/viz_5_top_brands.png')
plt.close()

print("EDA завершен. 5 графиков сохранены в data/processed/")

EDA завершен. 5 графиков сохранены в data/processed/
